# Week 41, Day 2: Amortised analysis

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w041d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def total_copies(n, grow):
    capacity, size, copies = 1, 0, 0
    for _ in range(n):
        if size == capacity:
            copies += size
            capacity = grow(capacity)
        size += 1
    return copies

for n in [1_000, 2_000, 4_000]:
    print(n, total_copies(n, lambda c: 2 * c), total_copies(n, lambda c: c + 100))

In [ ]:
def append_costs(n):
    capacity, size, costs = 1, 0, []
    for _ in range(n):
        cost = 1
        if size == capacity:
            cost += size
            capacity *= 2
        size += 1
        costs.append(cost)
    return costs

for n in [10, 100, 1000, 10_000]:
    costs = append_costs(n)
    print(f"n={n:>6}: most expensive {max(costs):>6}, total {sum(costs):>6}, per append {sum(costs) / n:.2f}")

In [ ]:
def bank(n, charge):
    capacity, size, balance, lowest = 1, 0, 0, 0
    for _ in range(n):
        balance += charge          # what this append is charged
        balance -= 1               # writing the item
        if size == capacity:
            balance -= size        # copying everything (before this item is added)
            capacity *= 2
        size += 1
        lowest = min(lowest, balance)
    return lowest

for charge in [1, 2, 3]:
    print(f"charge {charge}: lowest balance over 1000 appends = {bank(1000, charge)}")

In [ ]:
def increment(bits):
    """Add 1 to a list of bits, least significant first. Returns how many bits flipped."""
    i, flips = 0, 0
    while i < len(bits) and bits[i] == 1:
        bits[i] = 0
        flips += 1
        i += 1
    if i == len(bits):
        bits.append(1)
    else:
        bits[i] = 1
    return flips + 1

bits, total = [], 0
for n in range(1, 17):
    total += increment(bits)
    print(n, "".join(map(str, reversed(bits))), total)

> **Key idea.** Amortised cost is the total cost of any sequence of n operations divided by n: a guarantee for the whole sequence, not for each call. The aggregate method adds everything up; the accounting method charges each operation a fixed price and saves credit to pay for expensive ones. Doubling copies fewer than 2n items over n appends, so appends are O(1) amortised; growing by a constant copies about n²/(2k), so O(n) amortised. A binary counter and a stack with multi-pop are O(1) amortised too. Amortised is different from average, which depends on how the inputs are distributed.

### Exercise: A binary counter

Write `increment(bits)` for a list of bits stored **least significant first** (so `[1, 1, 0]` is 3): it adds 1 in place, appending a new 1 when it carries off the end, and returns how many bits changed. Then write `total_flips(n)`: the total number of bit changes for n increments starting from an empty list.

In [ ]:
def increment(bits):
    ___

def total_flips(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_increment():\n    \"incrementing\"\n    b = [1, 1, 0]\n    assert increment(b) == 3 and b == [0, 0, 1]\n    b = []\n    assert increment(b) == 1 and b == [1] and increment(b) == 2 and b == [0, 1]\n    b = [1, 1, 1]\n    assert increment(b) == 4 and b == [0, 0, 0, 1]\ndef test_total():\n    \"the total\"\n    assert [total_flips(n) for n in [1, 2, 3, 4, 8, 16]] == [1, 3, 4, 7, 15, 31]\n    for n in [10, 100, 1000, 5000]:\n        assert total_flips(n) < 2 * n")

### Exercise: A stack with multi-pop

Write `CountingStack` with `push(item)` (cost 1), `multipop(k)` (pops up to `k` items, or all if fewer, returning them in the order popped; cost 1 per item popped) and an attribute `cost`, the total cost of everything so far. Then write `check(ops)`, which runs a list of operations on a new stack (`("push", x)` or `("multipop", k)`) and returns `(cost, pushes)`.

In [ ]:
class CountingStack:
    ___

def check(ops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_stack():\n    \"multi-pop\"\n    s = CountingStack()\n    for x in \"abcde\":\n        s.push(x)\n    assert s.multipop(2) == [\"e\", \"d\"] and s.multipop(10) == [\"c\", \"b\", \"a\"] and s.multipop(3) == [] and s.cost == 10\ndef test_bound():\n    \"never more than twice the pushes\"\n    import random\n    rng = random.Random(4)\n    for _ in range(20):\n        ops = [(\"push\", i) if rng.random() < 0.7 else (\"multipop\", rng.randint(1, 20)) for i in range(300)]\n        cost, pushes = check(ops)\n        assert cost <= 2 * pushes\n    assert check([(\"push\", 1), (\"push\", 2), (\"multipop\", 5)]) == (4, 2)")

### Exercise: Is the charge enough?

Write `lowest_balance(n, charge, grow)`: simulate n appends into an array starting at capacity 1 that resizes to `grow(capacity)` when full. Each append adds `charge` to a balance, then pays 1 for writing, plus 1 for every item copied if it resized (the copy happens before the new item is written). Return the lowest balance reached (0 if it never drops below 0). Then write `smallest_charge(n, grow)`: the smallest whole-number charge from 1 upwards whose lowest balance over n appends is 0 or more.

In [ ]:
def lowest_balance(n, charge, grow):
    ___

def smallest_charge(n, grow):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_balance():\n    \"balances\"\n    double = lambda c: 2 * c\n    assert lowest_balance(1000, 3, double) == 0 and lowest_balance(1000, 2, double) < 0 and lowest_balance(0, 1, double) == 0\n    assert smallest_charge(1000, double) == 3 and smallest_charge(100_000, double) == 3\ndef test_constant():\n    \"growing by a constant needs a charge that grows with n\"\n    plus = lambda c: c + 10\n    small, big = smallest_charge(200, plus), smallest_charge(800, plus)\n    assert big > 3 * small > 0, (small, big)")

### Exercise: Comparing growth policies

Write `compare(n, policies)`, where `policies` is a dictionary from a name to a growth function. For each, simulate n appends from capacity 1 and return a dictionary from the name to `(copies, spare)`: the total items copied, and the empty slots left at the end (capacity minus n). Keep the names in the given order.

In [ ]:
def compare(n, policies):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\ndef test_compare():\n    \"policies\"\n    got = compare(1000, {\"double\": lambda c: 2 * c, \"half again\": lambda c: max(c + 1, math.ceil(1.5 * c)), \"plus 16\": lambda c: c + 16})\n    assert list(got) == [\"double\", \"half again\", \"plus 16\"]\n    assert got[\"double\"] == (1023, 24)\n    assert got[\"plus 16\"][0] > 5 * got[\"half again\"][0] and got[\"plus 16\"][1] < 16\n    assert compare(0, {\"x\": lambda c: 2 * c}) == {\"x\": (0, 1)}")

### Exercise: Thrashing

Write `resizes(ops, shrink_when)`: start an array at capacity 8 with 8 items, then apply a string of operations, `"a"` (append) and `"p"` (pop). An append into a full array doubles the capacity; after a pop, if the capacity is above 1 and `size <= capacity * shrink_when`, the capacity halves. Return the number of resizes. Then set `half` and `quarter` to the resizes for 100 alternating operations (`"ap" * 50`) with `shrink_when` of 0.5 and 0.25.

In [ ]:
def resizes(ops, shrink_when):
    ___

half = ___
quarter = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_thrash():\n    \"thrashing\"\n    assert resizes(\"\", 0.5) == 0 and resizes(\"a\", 0.25) == 1 and resizes(\"pppppp\", 0.25) == 1\n    assert half == 100 and quarter == 1")

### Bug hunt: A carry that never stops

`increment` should stop carrying at the first 0, but its loop condition only checks the length, so it turns **every** bit to 0: this prints `[0, 0, 0, 1]` for 3 + 1 (right) but `[0, 0, 0, 0, 1]` for 5 + 1 (wrong). Fix it so the second line is `[0, 1, 1]` (6).

In [ ]:
def increment(bits):
    i = 0
    while i < len(bits):
        bits[i] = 0
        i += 1
    if i == len(bits):
        bits.append(1)
    else:
        bits[i] = 1

a, b = [1, 1, 0], [1, 0, 1]
increment(a)
increment(b)
print(a)
print(b)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stop at the first 0\"\n    assert output.splitlines()[1] == \"[0, 1, 1]\", output\n    assert output.splitlines()[0] == \"[0, 0, 1]\"")

### Bug hunt: Growing by nothing

This array grows by half its capacity, but starting from capacity 1, `1 // 2` is 0, so it never grows and crashes with `IndexError`. Make it always grow by at least one slot, so this prints `10 13`.

In [ ]:
size, capacity, slots = 0, 1, [None]
for item in range(10):
    if size == capacity:
        new_capacity = capacity + capacity // 2
        new = [None] * new_capacity
        for i in range(size):
            new[i] = slots[i]
        slots, capacity = new, new_capacity
    slots[size] = item
    size += 1
print(size, capacity)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"always grow\"\n    expect_output(\"10 13\")")

### Bug hunt: Amortised, measured wrongly

This tries to show that appends cost O(1) amortised, but divides the **largest** single cost by n instead of the **total**, so it prints a number that shrinks towards 0 and proves nothing. Fix it so it prints the cost per append for each n: `2.5`, `2.023`, `2.023`.

In [ ]:
def append_costs(n):
    capacity, size, costs = 1, 0, []
    for _ in range(n):
        cost = 1
        if size == capacity:
            cost += size
            capacity *= 2
        size += 1
        costs.append(cost)
    return costs

for n in [10, 1000, 2000]:
    print(round(max(append_costs(n)) / n, 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"total divided by n\"\n    expect_output(\"2.5\\n2.023\\n2.023\")")

### Puzzle: The accounting method

Arrange the lines into a loop that checks a charge of 3 per append never runs out. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code.

p = ___   # total items copied by 64 appends into a doubling array starting at capacity 1
q = ___   # amortised cost of append when growing by a constant 1000 slots: "O(1)" or "O(n)"
r = ___   # bit flips to add 1 to binary 0111 (giving 1000)
s = ___   # most pops a sequence of 50 pushes and any number of multipops can do
t = ___   # True or False: an amortised O(1) operation is O(1) every single time

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 63 and q == \"O(n)\" and r == 4, \"p, q and r\"\n    assert s == 50 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a queue from two stacks

A queue (first in, first out) can be built from two stacks (Python lists used only with `append` and `pop()`): push new items onto an **in** stack; to dequeue, pop from an **out** stack, and only when the out stack is empty, move **every** item from in to out (which reverses them into the right order). One dequeue can move many items, but each item is moved at most once.

Write `TwoStackQueue` with `enqueue(item)`, `dequeue()` (raising `IndexError("dequeue from empty queue")`), `__len__`, and `moves`, the total number of items moved from in to out. Then write `worst_and_total(ops)`: run a string of operations (`"e"` enqueues the next whole number from 0 upwards; `"d"` dequeues) on a new queue, and return `(worst, total)`: the most items moved by any single dequeue, and the total moved. Use only `append` and `pop()` on the two lists.

In [ ]:
class TwoStackQueue:
    ___

def worst_and_total(ops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import ast\ndef test_queue():\n    \"first in, first out\"\n    q = TwoStackQueue()\n    for x in \"abc\":\n        q.enqueue(x)\n    assert q.dequeue() == \"a\" and len(q) == 2\n    q.enqueue(\"d\")\n    assert [q.dequeue(), q.dequeue(), q.dequeue()] == [\"b\", \"c\", \"d\"] and len(q) == 0 and q.moves == 4\n    try:\n        q.dequeue()\n        assert False\n    except IndexError as err:\n        assert str(err) == \"dequeue from empty queue\"\ndef test_amortised():\n    \"amortised\"\n    assert worst_and_total(\"e\" * 100 + \"d\" * 100) == (100, 100)\n    worst, total = worst_and_total(\"eed\" * 200 + \"d\" * 200)\n    assert total <= 400 and worst >= 2\n    assert worst_and_total(\"\") == (0, 0)\ndef test_only_stack_ops():\n    \"only append and pop()\"\n    for n in ast.walk(ast.parse(source)):\n        if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr == \"pop\":\n            assert not n.args, \"use pop() with no index: stacks only take from the top\"\n        if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute):\n            assert n.func.attr not in (\"insert\", \"reverse\"), \"only append and pop()\"")

## Warm-up

When n doubles, the copies with doubling roughly double too: the total is proportional to n. With "add 100 slots", the copies roughly **quadruple**: the total grows like n². Today is about proving both.

## One expensive step in many cheap ones

Three ways to describe an operation's cost:

| Kind | Question it answers | `append` on a doubling array |
|---|---|---|
| **worst case** | what's the most one call can cost? | O(n): the call that triggers a resize |
| **amortised** | over **any** sequence of n calls, what's the total, divided by n? | O(1) |
| **average** | over random inputs, what's the expected cost? | (a different question: needs probabilities) |

Amortised cost is a **guarantee**, not a hope: no matter what order the operations come in, n of them never cost more than n times the amortised cost. It says nothing about any single call, which is why a program that must answer within a fixed time, like one controlling a machine, sometimes can't use a dynamic array. For almost everything else, the total is what matters.

## The aggregate method

The simplest method: add up the cost of all n operations, and divide.

Count each append as 1 unit for writing the item, plus 1 unit for each item copied during a resize. With doubling from capacity 1, resizes happen when the size reaches 1, 2, 4, 8, ..., and copy that many items. For n appends, the copies are at most:

```text
1 + 2 + 4 + 8 + ... + (the largest power of 2 below n)   <   2n
```

(Each power of two is bigger than all the smaller ones put together, so the whole sum is less than twice the largest term, which is less than n.) So:

```text
total cost  <  n (writes)  +  2n (copies)  =  3n        amortised cost per append  <  3  =  O(1)
```

The most expensive single append grows with n, but the cost **per append** settles below 3.

## The accounting method

The aggregate method needs the whole sum. The **accounting** method (sometimes called the banker's method) works one operation at a time: charge every operation a fixed price, pay its real cost from that price, and **save the change** as credit for later. If the savings never go negative, the fixed price is an upper bound on the amortised cost.

For appends with doubling, charge **3 units** each:

- 1 pays for writing the new item;
- 2 are saved, "on" that item.

When the array resizes from capacity c to 2c, the c/2 items added since the last resize each carry 2 saved units: that's c units, exactly enough to copy all c items. So the bank never runs dry, and every append is paid for by its own fixed price of 3.

A charge of 2 isn't quite enough here, because the very first copies come before much has been saved; 3 never goes negative. (The method is credited to Robert Tarjan, whose 1985 paper "Amortized Computational Complexity" set out these techniques.)

## Growing by a constant

Suppose the array grows by a fixed k slots instead of doubling. Resizes happen at sizes k, 2k, 3k, ..., and copy that many items:

```text
k + 2k + 3k + ... + (n/k)·k   =   k · (1 + 2 + ... + n/k)   ≈   n² / (2k)
```

That's O(n²) in total, so **O(n) amortised** per append, however big k is: a bigger k only divides the cost by a constant. Growing by a **factor** (any factor above 1) is what makes appends O(1) amortised. Different languages choose different factors:

| Growth | Copies for n appends | Wasted space just after growing |
|---|---|---|
| + k slots | about n²/(2k): O(n) amortised | at most k slots |
| × 1.5 | about 2n | up to a third of the block |
| × 2 | about n | up to half of the block |

A smaller factor wastes less memory and copies a little more; both are O(1) amortised. CPython's list grows by about an eighth plus a little; that's still a factor, so still O(1) amortised.

## More amortised structures

The same reasoning works far beyond arrays.

**A binary counter.** Adding 1 to a binary number flips the trailing 1s to 0s and one 0 to 1. Adding 1 to `0111` flips four bits; adding 1 to `1000` flips one. The worst case is O(number of bits), but bit 0 flips every time, bit 1 every second time, bit 2 every fourth time... so n increments flip fewer than `n + n/2 + n/4 + ... < 2n` bits: O(1) amortised.

**A stack with multi-pop.** If a stack has `push` (cost 1) and `multipop(k)` (cost up to k), one multipop can be expensive, but every item can only be popped once after being pushed once. So any sequence of n operations costs at most 2n: O(1) amortised each. The accounting view: charge each push 2, and the saved 1 pays for that item's eventual pop.

**Amortised, not average.** These results hold for **every** sequence of operations: no luck involved. Average-case analysis (Week 17) is different: it assumes something about the inputs, such as random order, and can be beaten by unlucky or hostile inputs. Python's dictionaries are O(1) on **average**; its list appends are O(1) **amortised**.

## Practice

## Quiz

## Daily challenge

## Before you go

Explain to yourself, without notes, why `n` appends with doubling copy fewer than `2n` items. Tomorrow: two pointers, a pattern that turns many O(n²) problems on arrays into O(n).